# Moirai VaR-aware Training on Kaggle
Notebook này được cấu hình sẵn để chạy full-pipeline huấn luyện mô hình Moirai VaR-aware.

**Yêu cầu môi trường:**
- Dataset: Mount tại `/kaggle/input/datasets/trnhngv/historical-price`
- Weights: Mount tại `/kaggle/input/datasets/trnhngv/new-weights/weights`
- GPU: Hãy bật T4 x2 hoặc P100 trên Kaggle.
- Internet: Bật Always ON trên Kaggle.

In [ ]:
# 1. Setup Môi trường: Clone Source Code từ đúng branch analysis_data
!rm -rf /kaggle/working/repo
!git clone -b analysis_data https://github.com/nhanbayern/1003_EPA-Project_UIT.git /kaggle/working/repo

# Clone thư viện uni2ts từ SalesforceAIResearch
!rm -rf /kaggle/working/uni2ts
!git clone https://github.com/SalesforceAIResearch/uni2ts.git /kaggle/working/uni2ts

# Cài đặt các thư viện cần thiết
!pip install -q gluonts torch pandas numpy scipy jaxtyping "jax[cpu]" hydra-core huggingface_hub


In [ ]:
# 2. Cấu hình Python Path
import sys
import os

# Đưa thư mục repo vào sys.path để import được các module trong experiments/
sys.path.insert(0, '/kaggle/working/repo')

# Đưa thư mục uni2ts vừa clone vào sys.path
sys.path.insert(0, '/kaggle/working/uni2ts/src')

In [ ]:
# 3. Import các thư viện và hàm cốt lõi
import torch
import pandas as pd
from torch.utils.data import DataLoader

from experiments.moirai_var_aware.config import SPLIT_INFO
from experiments.moirai_var_aware.data import load_and_split_dataset
from experiments.moirai_var_aware.export import build_prediction_frame
from experiments.moirai_var_aware.modeling import (
    VolatilityFeatureExtractor,
    VolatilityRegressionModel,
    patch_uni2ts_exports,
)
from experiments.moirai_var_aware.train import predict, train_model_var_aware

# Patch thư viện uni2ts với đường dẫn cụ thể trên Kaggle
patch_uni2ts_exports(base='/kaggle/working/uni2ts/src/uni2ts/model')

In [ ]:
# 4. Cấu hình Tham số Huấn luyện
import os
DATASET_DIR = '/kaggle/input/datasets/trnhngv/historical-price'

# Tự động tìm đường dẫn weights hợp lệ từ danh sách ứng viên (ưu tiên new-weights của bạn)
WEIGHTS_CANDIDATES = [
    '/kaggle/input/datasets/trnhngv/new-weights/weights',
    '/kaggle/input/new-weights/weights',
    '/kaggle/input/new_weigths/weights',
    '/kaggle/input/new_weigths',
    '/kaggle/input/datasets/trnhngv/weights',
    '/kaggle/working/weights',
]

WEIGHTS_DIR = '/kaggle/input/datasets/trnhngv/new-weights/weights'
for p in WEIGHTS_CANDIDATES:
    if os.path.exists(os.path.join(p, 'moirai-1.1-R-small')):
        WEIGHTS_DIR = p
        break

print(f"Dataset dir : {DATASET_DIR}")
print(f"Weights dir : {WEIGHTS_DIR} (Ton tai: {os.path.exists(WEIGHTS_DIR)})")

MODELS_TO_TRAIN = ["moirai", "moirai2", "moirai_moe"]
DATASETS_TO_TRAIN = list(SPLIT_INFO.keys())  # Bạn có thể chỉ định riêng mảng, vd: ["VN_INDEX"]

LAMBDA_VARS = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
EPOCHS = 30
BATCH_SIZE = 8  # Chỉnh nhỏ lại (vd: 4 hoặc 8) nếu bị lỗi OOM (Out of Memory) trên Kaggle
TUNING_MODE = "head"  # Các tuỳ chọn: "head", "full"
BACKBONE_LR = 1e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dang su dung thiet bi:", device)


In [ ]:
# 5. Vòng lặp Training (Full Pipeline)
import numpy as np
from scipy.stats import kurtosis

all_frames = []
csv_outputs = {} # Lưu trữ nội dung CSV gốc theo tên file

for lambda_var in LAMBDA_VARS:
    print(f"\n" + "="*50)
    print(f"BẮT ĐẦU HUẤN LUYỆN VỚI LAMBDA_VAR = {lambda_var}")
    print("="*50 + "\n")
    for index_name in DATASETS_TO_TRAIN:
        csv_path = os.path.join(DATASET_DIR, f"{index_name}.csv")
        if not os.path.exists(csv_path):
            print(f"Bỏ qua {index_name} vì không tìm thấy file: {csv_path}")
            continue

        print(f"\n========== Đang xử lý Dataset: {index_name} ==========")
        train_ds, val_ds, test_ds = load_and_split_dataset(csv_path, index_name)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

        # Tính toán dynamic_nu
        train_returns = np.array([train_ds.dataset.samples[i]["log_return"] for i in train_ds.indices])
        k = kurtosis(train_returns, fisher=True, nan_policy="omit")
        if k <= 0:
            dynamic_nu = 30.0
        else:
            dynamic_nu = float(np.clip(4.0 + 6.0 / k, 2.1, 30.0))
        print(f"[{index_name}] Kurtosis: {k:.4f} | Calculated Dynamic nu: {dynamic_nu:.4f}")

        for model_type in MODELS_TO_TRAIN:
            print(f"\n--- {index_name} | Model: {model_type} | Lambda: {lambda_var} | Mode: {TUNING_MODE} | nu: {dynamic_nu:.4f} ---")
            
            extractor = VolatilityFeatureExtractor(
                model_type=model_type,
                size="small",
                device=device,
                weights_dir=WEIGHTS_DIR,
                freeze_backbone=(TUNING_MODE == "head"),
            )
            model = VolatilityRegressionModel(extractor=extractor)
            
            # Bắt đầu quá trình học
            model = train_model_var_aware(
                model,
                train_loader,
                val_loader,
                epochs=EPOCHS,
                lr=1e-3,
                backbone_lr=BACKBONE_LR,
                device=device,
                alpha=0.01,
                lambda_var=lambda_var,
                distribution="student_t",
                nu=dynamic_nu,
                var_horizon_index=0,
                tuning_mode=TUNING_MODE,
            )
            
            # Dự đoán trên tập Test
            # Validation predictions are retained for a leakage-free rescaling control.
            val_preds, val_targets = predict(model, val_loader, device=device)
            preds, targets = predict(model, test_loader, device=device)
            
            # Lấy kết quả trả về từ hàm build_prediction_frame
            val_frame = build_prediction_frame(
                index_name, model_type, lambda_var, val_ds, val_preds, val_targets, tuning_mode=TUNING_MODE, split='validation'
            )
            frame = build_prediction_frame(
                index_name, model_type, lambda_var, test_ds, preds, targets, tuning_mode=TUNING_MODE
            )
            all_frames.append(frame)
            
            # Định dạng tên file theo đúng chuẩn của file MD gốc
            mode_suffix = "" if TUNING_MODE == "head" else f"_{TUNING_MODE}"
            filename = f"{index_name}_{model_type}{mode_suffix}_lambda_{lambda_var:g}_predictions.csv"
            csv_outputs[filename] = frame.to_csv(index=False)
            csv_outputs[filename.replace('_predictions.csv', '_validation_predictions.csv')] = val_frame.to_csv(index=False)
            
            # Giải phóng bộ nhớ RAM/VRAM để train model tiếp theo
            del model
            del extractor
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


In [ ]:
# 6. Lưu kết quả gốc thành file Zip (Output chuẩn theo document)
import zipfile

if len(csv_outputs) > 0:
    mode_str = "" if TUNING_MODE == "head" else "_full"
    # Tên file zip gộp tất cả các lambda
    zip_filename = f"/kaggle/working/moirai_var{mode_str}_multi_lambda_predictions.zip"
    
    with zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for filename, content in csv_outputs.items():
            zf.writestr(filename, content)
    print(f"✅ Đã tạo file Zip nguyên bản tại: {zip_filename} (Chứa {len(csv_outputs)} file csv)")
else:
    print("Không có kết quả gốc nào để nén Zip.")